# Module 4: Medallion Pipelines — Bronze, Silver, Gold

This notebook demonstrates the **medallion architecture** for ETL:
- **Bronze**: Raw ingestion with minimal transforms
- **Silver**: Conformed dimensions and facts
- **Gold**: Business-ready KPI aggregates

Run each cell in order to build your medallion pipeline.

## Setup: Ensure we have the HR dataset

In [ ]:
# Set the database for this notebook
spark.sql("USE workspace.default")

# Check if hr_dataset exists
spark.sql("SELECT COUNT(*) as record_count FROM hr_dataset").show()

## Part 1: Bronze Layer — Raw Ingestion

In [ ]:
from pyspark.sql import functions as F

# Create the bronze table: raw data with minimal schema fixes
bronze_df = spark.read.table("workspace.default.hr_dataset") \
    .withColumnRenamed("EDUCATION LEVEL", "EDUCATION_LEVEL")

# Write as a managed Delta table
bronze_df.write.mode("overwrite").option("mergeSchema", "true").saveAsTable("bronze_hr_dataset")

print("✓ Bronze table created: bronze_hr_dataset")
spark.sql("SELECT COUNT(*) as rows FROM bronze_hr_dataset").show()

## Part 2: Silver Layer — Conformed Dimensions and Facts

### Dimension: dim_employee

In [ ]:
# Create dim_employee: unique employee attributes
dim_employee = spark.read.table("bronze_hr_dataset") \
    .select(
        "EMP_ID",
        "EMP_NM",
        "EMAIL",
        "EDUCATION_LEVEL"
    ) \
    .distinct()

dim_employee.write.mode("overwrite").option("mergeSchema", "true").saveAsTable("dim_employee")

print("✓ Dimension created: dim_employee")
spark.sql("SELECT COUNT(*) as employee_count FROM dim_employee").show()
spark.sql("SELECT * FROM dim_employee LIMIT 3").show()

### Dimension: dim_department

In [ ]:
# Create dim_department: unique departments with managers
dim_department = spark.read.table("bronze_hr_dataset") \
    .select("DEPARTMENT", "MANAGER") \
    .distinct()

dim_department.write.mode("overwrite").option("mergeSchema", "true").saveAsTable("dim_department")

print("✓ Dimension created: dim_department")
spark.sql("SELECT * FROM dim_department").show()

### Dimension: dim_location

In [ ]:
# Create dim_location: unique locations
dim_location = spark.read.table("bronze_hr_dataset") \
    .select("LOCATION") \
    .filter(F.col("LOCATION").isNotNull()) \
    .distinct()

dim_location.write.mode("overwrite").option("mergeSchema", "true").saveAsTable("dim_location")

print("✓ Dimension created: dim_location")
spark.sql("SELECT * FROM dim_location").show()

### Dimension: dim_date

In [ ]:
# Create dim_date: derived from all date columns
dim_date = spark.read.table("bronze_hr_dataset") \
    .select(F.col("HIRE_DT").alias("date_col")) \
    .union(
        spark.read.table("bronze_hr_dataset") \
        .select(F.col("START_DT").alias("date_col"))
    ) \
    .union(
        spark.read.table("bronze_hr_dataset") \
        .select(F.col("END_DT").alias("date_col"))
    ) \
    .filter(F.col("date_col").isNotNull()) \
    .select(
        F.col("date_col").cast("date").alias("date"),
        F.year("date_col").alias("year"),
        F.month("date_col").alias("month"),
        F.quarter("date_col").alias("quarter"),
        F.dayofmonth("date_col").alias("day"),
        F.dayofweek("date_col").alias("day_of_week"),
        F.weekofyear("date_col").alias("week_of_year")
    ) \
    .distinct()

dim_date.write.mode("overwrite").option("mergeSchema", "true").saveAsTable("dim_date")

print("✓ Dimension created: dim_date")
spark.sql("SELECT COUNT(*) as date_count FROM dim_date").show()
spark.sql("SELECT * FROM dim_date ORDER BY date LIMIT 5").show()

### Fact Table: fact_employee

In [ ]:
# Create fact_employee: employee facts with foreign keys to dimensions
fact_employee = spark.read.table("bronze_hr_dataset") \
    .select(
        "EMP_ID",
        "JOB_TITLE",
        "DEPARTMENT",
        "LOCATION",
        F.col("HIRE_DT").cast("date").alias("HIRE_DT"),
        F.col("START_DT").cast("date").alias("START_DT"),
        F.col("END_DT").cast("date").alias("END_DT"),
        "SALARY"
    )

fact_employee.write.mode("overwrite").option("mergeSchema", "true").saveAsTable("fact_employee")

print("✓ Fact table created: fact_employee")
spark.sql("SELECT COUNT(*) as fact_rows FROM fact_employee").show()
spark.sql("SELECT * FROM fact_employee LIMIT 3").show()

## Part 3: Gold Layer — Business-Ready KPIs

### Gold: Company-Wide Summary

In [ ]:
# Create gold_company_summary: organization-wide KPIs (single row)
gold_company_summary = spark.read.table("fact_employee") \
    .agg(
        F.lit("Company Wide").alias("METRIC_LEVEL"),
        F.current_timestamp().alias("SNAPSHOT_TIMESTAMP"),
        F.count("EMP_ID").alias("TOTAL_EMPLOYEES"),
        F.countDistinct("DEPARTMENT").alias("TOTAL_DEPARTMENTS"),
        F.countDistinct("LOCATION").alias("TOTAL_LOCATIONS"),
        F.countDistinct("JOB_TITLE").alias("TOTAL_JOB_TITLES"),
        F.sum("SALARY").alias("TOTAL_PAYROLL"),
        F.avg("SALARY").alias("AVG_SALARY"),
        F.percentile_approx("SALARY", 0.5).alias("MEDIAN_SALARY"),
        F.min("SALARY").alias("MIN_SALARY"),
        F.max("SALARY").alias("MAX_SALARY"),
        F.min("HIRE_DT").alias("EARLIEST_HIRE_DATE"),
        F.max("HIRE_DT").alias("LATEST_HIRE_DATE")
    ) \
    .withColumn(
        "AVG_SALARY_PER_DEPARTMENT",
        F.col("TOTAL_PAYROLL") / F.col("TOTAL_DEPARTMENTS")
    ) \
    .withColumn(
        "SALARY_RANGE",
        F.col("MAX_SALARY") - F.col("MIN_SALARY")
    )

gold_company_summary.write.mode("overwrite").option("mergeSchema", "true").saveAsTable("gold_company_summary")

print("✓ Gold table created: gold_company_summary")
spark.sql("SELECT * FROM gold_company_summary").show(vertical=True)

### Gold: Department Metrics

In [ ]:
# Create gold_department_metrics: KPIs per department
gold_department_metrics = spark.read.table("fact_employee") \
    .groupBy("DEPARTMENT") \
    .agg(
        F.count("EMP_ID").alias("EMPLOYEE_COUNT"),
        F.sum("SALARY").alias("TOTAL_SALARY"),
        F.avg("SALARY").alias("AVG_SALARY"),
        F.min("SALARY").alias("MIN_SALARY"),
        F.max("SALARY").alias("MAX_SALARY"),
        F.percentile_approx("SALARY", 0.5).alias("MEDIAN_SALARY"),
        F.countDistinct("JOB_TITLE").alias("UNIQUE_JOB_TITLES"),
        F.countDistinct("LOCATION").alias("UNIQUE_LOCATIONS"),
        F.min("HIRE_DT").alias("EARLIEST_HIRE_DATE"),
        F.max("HIRE_DT").alias("LATEST_HIRE_DATE")
    )

gold_department_metrics.write.mode("overwrite").option("mergeSchema", "true").saveAsTable("gold_department_metrics")

print("✓ Gold table created: gold_department_metrics")
spark.sql("SELECT * FROM gold_department_metrics ORDER BY TOTAL_SALARY DESC").show()

### Gold: Hiring Trends

In [ ]:
# Create gold_hiring_trends: temporal hiring patterns
gold_hiring_trends = spark.read.table("fact_employee") \
    .withColumn("HIRE_YEAR", F.year("HIRE_DT")) \
    .withColumn("HIRE_MONTH", F.month("HIRE_DT")) \
    .withColumn("HIRE_QUARTER", F.quarter("HIRE_DT")) \
    .groupBy("HIRE_YEAR", "HIRE_QUARTER", "HIRE_MONTH", "DEPARTMENT") \
    .agg(
        F.count("EMP_ID").alias("HIRES_COUNT"),
        F.avg("SALARY").alias("AVG_STARTING_SALARY"),
        F.countDistinct("JOB_TITLE").alias("UNIQUE_ROLES_HIRED")
    ) \
    .withColumn(
        "YEAR_MONTH",
        F.concat(
            F.col("HIRE_YEAR"),
            F.lit("-"),
            F.lpad(F.col("HIRE_MONTH"), 2, "0")
        )
    )

gold_hiring_trends.write.mode("overwrite").option("mergeSchema", "true").saveAsTable("gold_hiring_trends")

print("✓ Gold table created: gold_hiring_trends")
spark.sql("SELECT YEAR_MONTH, DEPARTMENT, HIRES_COUNT, AVG_STARTING_SALARY FROM gold_hiring_trends ORDER BY YEAR_MONTH DESC, HIRES_COUNT DESC LIMIT 10").show()

### Gold: Location Analytics

In [ ]:
from pyspark.sql.window import Window

# Create gold_location_analytics: location-based insights with ranking
gold_location_analytics = spark.read.table("fact_employee") \
    .groupBy("LOCATION") \
    .agg(
        F.count("EMP_ID").alias("EMPLOYEE_COUNT"),
        F.countDistinct("DEPARTMENT").alias("DEPARTMENTS_COUNT"),
        F.countDistinct("JOB_TITLE").alias("JOB_TITLES_COUNT"),
        F.avg("SALARY").alias("AVG_SALARY"),
        F.sum("SALARY").alias("TOTAL_SALARY"),
        F.percentile_approx("SALARY", 0.5).alias("MEDIAN_SALARY"),
        F.stddev("SALARY").alias("SALARY_STDDEV")
    ) \
    .withColumn(
        "SALARY_COST_RANK",
        F.dense_rank().over(
            Window.orderBy(F.desc("TOTAL_SALARY"))
        )
    ) \
    .withColumn(
        "HEADCOUNT_RANK",
        F.dense_rank().over(
            Window.orderBy(F.desc("EMPLOYEE_COUNT"))
        )
    )

gold_location_analytics.write.mode("overwrite").option("mergeSchema", "true").saveAsTable("gold_location_analytics")

print("✓ Gold table created: gold_location_analytics")
spark.sql("SELECT * FROM gold_location_analytics ORDER BY HEADCOUNT_RANK").show()

## Part 4: Verify the Complete Pipeline

In [ ]:
# List all tables by layer
print("=== MEDALLION PIPELINE SUMMARY ===")
print("\n🥉 BRONZE (Raw):")
spark.sql("SHOW TABLES LIKE 'bronze*'").show()

print("\n🥈 SILVER (Conformed):")
spark.sql("SHOW TABLES LIKE 'dim*' OR SHOW TABLES LIKE 'fact*'").show()

print("\n🥇 GOLD (Business Ready):")
spark.sql("SHOW TABLES LIKE 'gold*'").show()

## Part 5: Example BI Queries on Gold Tables

### Query 1: Which department has the highest total payroll?

In [ ]:
spark.sql("""
    SELECT 
        DEPARTMENT,
        EMPLOYEE_COUNT,
        TOTAL_SALARY as PAYROLL,
        AVG_SALARY,
        MEDIAN_SALARY
    FROM gold_department_metrics
    ORDER BY PAYROLL DESC
    LIMIT 5
""").show()

### Query 2: Hiring trends by quarter

In [ ]:
spark.sql("""
    SELECT 
        YEAR_MONTH,
        DEPARTMENT,
        HIRES_COUNT,
        AVG_STARTING_SALARY,
        UNIQUE_ROLES_HIRED
    FROM gold_hiring_trends
    ORDER BY YEAR_MONTH DESC, HIRES_COUNT DESC
    LIMIT 15
""").show()

### Query 3: Location analysis with rankings

In [ ]:
spark.sql("""
    SELECT 
        LOCATION,
        EMPLOYEE_COUNT,
        HEADCOUNT_RANK,
        TOTAL_SALARY,
        SALARY_COST_RANK,
        AVG_SALARY,
        MEDIAN_SALARY
    FROM gold_location_analytics
    ORDER BY HEADCOUNT_RANK
""").show()

## Mini-Assignment

1. **Data Quality**: What data quality checks would you add to the bronze layer? (E.g., reject rows where EMP_ID is null)
2. **CFO Query**: Which gold table would a CFO query first, and why?
3. **Schema Changes**: If the source HR dataset added a new column (e.g., `HIRE_REASON`), which layers would you need to update?
4. **PySpark Exercise**: Write a 4-line query that reads `gold_department_metrics` and finds the top 3 departments by AVG_SALARY.

In [ ]:
# Write your PySpark solution here
top_3_salary_depts = spark.read.table("gold_department_metrics") \
    .select("DEPARTMENT", "AVG_SALARY", "EMPLOYEE_COUNT") \
    .orderBy(F.col("AVG_SALARY").desc()) \
    .limit(3)

top_3_salary_depts.show()